# 03 — VL-JEPA Pretraining

Self-Supervised Representation Learning on 3D CT patches.

Pipeline:
context → online encoder → predictor → z_pred
target → momentum encoder → z_target (stop-grad)

Loss:
SmoothL1(z_pred, z_target)

EMA update for momentum encoder.

No labels used.


In [6]:
import sys
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

# If running from notebooks folder, use this instead:
# PROJECT_ROOT = Path().resolve().parents[1]
# sys.path.append(str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)


Project root added: /home/sreethanu/lidc_ldri_dataset_1/lung_cancer_vl_jepa_lidc


In [7]:
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from torch.utils.data import DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

from src.models.encoder_3d import ViT3DEncoder
from src.models.momentum_encoder import MomentumEncoder, get_momentum_schedule
from src.models.jepa_predictor import JEPAPredictor
from src.models.masking import BlockMask3D
from src.utils.losses import JEPALoss
from src.utils.config import Config
from src.utils.seed import set_global_seed
from src.utils.ssl_dataset import SSLDataset


In [8]:
config_path = PROJECT_ROOT / "configs" / "jepa_config.yaml"
config = Config(str(config_path))

set_global_seed(
    config.get("project.seed"),
    deterministic=config.get("project.deterministic")
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Setting global seed: 42
Enabling deterministic mode (may reduce performance)
Reproducibility configured successfully.
Device: cuda


In [9]:
patch_root = PROJECT_ROOT / "data" / "processed" / "patches"
patch_paths = list(patch_root.rglob("*.npz"))

print("Total patches found:", len(patch_paths))

dataset = SSLDataset(patch_paths)

val_size = int(0.1 * len(dataset))
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=config.get("data.batch_size"),
    shuffle=True,
    num_workers=config.get("data.num_workers"),
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.get("data.batch_size"),
    shuffle=False,
    num_workers=config.get("data.num_workers"),
    pin_memory=True,
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


Total patches found: 3054
Train batches: 459
Val batches: 51


In [10]:
online_encoder = ViT3DEncoder(
    input_size=tuple(config.get("data.input_shape")[1:]),
    patch_size=tuple(config.get("model.encoder.patch_size")),
    embed_dim=config.get("model.encoder.embed_dim"),
    depth=config.get("model.encoder.depth"),
    num_heads=config.get("model.encoder.num_heads"),
).to(device)

target_encoder = MomentumEncoder(
    base_encoder=online_encoder,
    momentum=config.get("model.jepa.ema_decay_start")
).to(device)

predictor = JEPAPredictor(
    embed_dim=config.get("model.encoder.embed_dim"),
    predictor_embed_dim=config.get("model.predictor.embed_dim"),
    depth=config.get("model.predictor.depth"),
    num_heads=config.get("model.predictor.num_heads"),
).to(device)

print("Online encoder params:", sum(p.numel() for p in online_encoder.parameters()))
print("Predictor params:", sum(p.numel() for p in predictor.parameters()))


Online encoder params: 88368384
Predictor params: 11238912


In [11]:
masker = BlockMask3D(
    input_size=tuple(config.get("data.input_shape")[1:]),
    patch_size=tuple(config.get("model.encoder.patch_size")),
    num_blocks=config.get("model.masking.num_blocks"),
    min_block_scale=config.get("model.masking.min_block_scale"),
    max_block_scale=config.get("model.masking.max_block_scale"),
)

criterion = JEPALoss(loss_type="smooth_l1")


In [12]:
params = list(online_encoder.parameters()) + list(predictor.parameters())

optimizer = torch.optim.AdamW(
    params,
    lr=config.get("pretraining.learning_rate"),
    weight_decay=config.get("pretraining.weight_decay"),
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.get("pretraining.epochs"),
    eta_min=config.get("pretraining.min_lr"),
)

momentum_schedule = get_momentum_schedule(
    base_momentum=config.get("model.jepa.ema_decay_start"),
    final_momentum=config.get("model.jepa.ema_decay_end"),
    epochs=config.get("pretraining.epochs"),
    warmup_epochs=config.get("model.jepa.ema_warmup_epochs"),
)

scaler = GradScaler(enabled=config.get("pretraining.amp"))


In [13]:
volumes = next(iter(train_loader))
volumes = volumes.to(device)

visible_mask, target_mask = masker(volumes.size(0))
visible_mask = visible_mask.to(device)
target_mask = target_mask.to(device)

with torch.no_grad():
    z_context = online_encoder(volumes, mask=visible_mask)
    z_target = target_encoder(volumes)

print("Context shape:", z_context.shape)
print("Target shape:", z_target.shape)
print("Total tokens:", online_encoder.get_num_patches())


Context shape: torch.Size([6, 216, 768])
Target shape: torch.Size([6, 216, 768])
Total tokens: 216


In [15]:
save_dir = PROJECT_ROOT / "checkpoints" / "pretraining"
save_dir.mkdir(parents=True, exist_ok=True)

epochs = config.get("pretraining.epochs")
save_freq = config.get("pretraining.save_freq")

print("Starting JEPA pretraining...")

for epoch in range(epochs):

    online_encoder.train()
    predictor.train()

    total_loss = 0.0

    for volumes in tqdm(train_loader, desc=f"Epoch {epoch+1}"):

        volumes = volumes.to(device, non_blocking=True)
        B = volumes.size(0)

        visible_mask, target_mask = masker(B)
        visible_mask = visible_mask.to(device)
        target_mask = target_mask.to(device)

        optimizer.zero_grad()

        with autocast(enabled=config.get("pretraining.amp")):

            # Online path (masked)
            z_context = online_encoder(volumes, mask=visible_mask)

            # Target path (no grad)
            with torch.no_grad():
                z_target = target_encoder(volumes)

            # Predictor
            z_pred = predictor(
                z_context,
                target_mask,
                online_encoder.get_num_patches()
            )

            # Extract masked tokens
            masked_targets = []
            masked_preds = []
            
            for b in range(B):
                tgt = z_target[b][target_mask[b]]
                pred = z_pred[b]
            
                min_len = min(tgt.size(0), pred.size(0))
            
                masked_targets.append(tgt[:min_len])
                masked_preds.append(pred[:min_len])
            
            # 🔥 Concatenate instead of stack
            masked_targets = torch.cat(masked_targets, dim=0)
            masked_preds = torch.cat(masked_preds, dim=0)
            
            loss = criterion(masked_preds, masked_targets)


            loss = criterion(masked_preds, masked_targets)

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(
            params,
            config.get("pretraining.gradient_clip_val")
        )

        scaler.step(optimizer)
        scaler.update()

        # EMA update
        momentum = momentum_schedule[epoch]
        target_encoder.update(online_encoder, momentum)

        total_loss += loss.item()

    scheduler.step()

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch [{epoch+1}/{epochs}] | Loss: {avg_loss:.6f}")

    if (epoch + 1) % save_freq == 0:
        torch.save(
            {
                "epoch": epoch,
                "online_encoder": online_encoder.state_dict(),
                "target_encoder": target_encoder.encoder.state_dict(),
                "predictor": predictor.state_dict(),
                "optimizer": optimizer.state_dict(),
            },
            save_dir / f"jepa_epoch_{epoch+1}.pth"
        )

        print("Checkpoint saved.")


Starting JEPA pretraining...


Epoch 1: 100%|████████████████████████████████| 459/459 [01:16<00:00,  5.98it/s]



Epoch [1/100] | Loss: 0.079867


Epoch 2: 100%|████████████████████████████████| 459/459 [01:07<00:00,  6.78it/s]



Epoch [2/100] | Loss: 0.124440


...


Epoch [98/100] | Loss: 0.121861


Epoch 99: 100%|███████████████████████████████| 459/459 [01:11<00:00,  6.40it/s]



Epoch [99/100] | Loss: 0.122143


Epoch 100: 100%|██████████████████████████████| 459/459 [01:01<00:00,  7.52it/s]



Epoch [100/100] | Loss: 0.122259
Checkpoint saved.


In [16]:
from pathlib import Path
import os

save_dir = PROJECT_ROOT / "checkpoints" / "pretraining"

print("Checking directory:", save_dir)

if save_dir.exists():
    print("✔ Pretraining directory exists.")
    
    checkpoint_files = list(save_dir.glob("*.pth"))
    
    if len(checkpoint_files) == 0:
        print("⚠ No checkpoint files found.")
    else:
        print(f"✔ Found {len(checkpoint_files)} checkpoint file(s):")
        for file in checkpoint_files:
            size_mb = os.path.getsize(file) / (1024 * 1024)
            print(f"   - {file.name} | {size_mb:.2f} MB")

else:
    print("❌ Pretraining directory does NOT exist.")


Checking directory: /home/sreethanu/lidc_ldri_dataset_1/lung_cancer_vl_jepa_lidc/checkpoints/pretraining
✔ Pretraining directory exists.
✔ Found 10 checkpoint file(s):
   - jepa_epoch_10.pth | 1477.34 MB
   - jepa_epoch_70.pth | 1477.34 MB
   - jepa_epoch_60.pth | 1477.34 MB
   - jepa_epoch_90.pth | 1477.34 MB
   - jepa_epoch_30.pth | 1477.34 MB
   - jepa_epoch_20.pth | 1477.34 MB
   - jepa_epoch_80.pth | 1477.34 MB
   - jepa_epoch_50.pth | 1477.34 MB
   - jepa_epoch_40.pth | 1477.34 MB
   - jepa_epoch_100.pth | 1477.34 MB


In [17]:
best_model_path = save_dir / "best_model.pth"

if best_model_path.exists():
    size_mb = os.path.getsize(best_model_path) / (1024 * 1024)
    print(f"✔ best_model.pth exists | {size_mb:.2f} MB")
else:
    print("⚠ best_model.pth not found.")


⚠ best_model.pth not found.


In [18]:
results_dir = PROJECT_ROOT / "results" / "pretraining"

if results_dir.exists():
    print("✔ Results directory exists.")
    print("Files inside:", [f.name for f in results_dir.iterdir()])
else:
    print("ℹ No results directory found (optional).")


ℹ No results directory found (optional).


In [19]:
checkpoint_path = PROJECT_ROOT / "checkpoints" / "pretraining" / "jepa_epoch_100.pth"

checkpoint = torch.load(checkpoint_path, map_location=device)

online_encoder.load_state_dict(checkpoint["online_encoder"])
online_encoder.eval()

print("Pretrained encoder (epoch 100) loaded successfully.")


Pretrained encoder (epoch 100) loaded successfully.
